In [0]:
%run "../00_config"

In [0]:
import random
from datetime import datetime, timezone
from pyspark.sql import functions as F

##Load bronze products

In [0]:
df_products = (spark.table(BRONZE_PRODUCTS)
   .select("asin", "current_price_sar", "category")
   .filter(F.col("asin") != "")
   .filter(F.col("current_price_sar") != 0)
   .filter(F.col("current_price_sar").isNotNull())
   .dropDuplicates(["asin"])
)
products_list = df_products.collect()
print(f"Available products: {len(products_list)}")
print(f"Sample: {products_list[0]}")

##Load bronze orders

In [0]:
df_orders = (spark.table(BRONZE_ORDERS))
print(f"Orders enrich: {df_orders.count()}")

##Inject products filed to orders

In [0]:
orders_list = df_orders.collect()
enriched_orders = []
for row in orders_list:
   order = row.asDict()
   product = random.choice(products_list)
   order["product_id"]       = product["asin"]
   order["unit_price_sar"]   = product["current_price_sar"]
   enriched_orders.append(order)
print(f"Enriched orders: {len(enriched_orders)}")
print(f"\nSample:")
print(f"  product_id      : {enriched_orders[0]['product_id']}")
print(f"  unit_price_sar  : {enriched_orders[0]['unit_price_sar']}")

##Write to Bronze orders enriched

In [0]:
df_enriched = spark.createDataFrame(enriched_orders)
(df_enriched
   .write
   .format("delta")
   .mode("overwrite")
   .saveAsTable(BRONZE_ORDERS_ENRICHED)
)
print(f"✅ {df_enriched.count()} enriched orders written to {BRONZE_ORDERS_ENRICHED}")

##Validate

In [0]:
df_check = spark.table(BRONZE_ORDERS_ENRICHED)
print(f"Total rows : {df_check.count()}")
print(f"Columns    : {df_check.columns}")
df_check.select(
   "order_id", "city", "product_id",
   "unit_price_sar", "order_status"
).show(5, truncate=35)

In [0]:
%sql
SELECT * FROM saudi_retail_catalog.bronze.bronze_orders_enriched